# quantum-cortex — REPRISE step 4: the journal in the decode loop, fine tune from N1, the LM arm of the H.M. protocol (ADR-008)

**Settings:** Accelerator = GPU T4, Internet = On. **Inputs:** the N1 notebook's version whose Output holds `runs/kaggle_t4/ckpt.pt` (the parent) and, if present, `data/fineweb_edu_bytes.bin` (the same slice; else it is rebuilt). **Secret:** `QUANTUM_CORTEX_JOURNAL_KEY`, 32 bytes hex, generated once locally with `python -c "from cortex_c2b.crypto import generate_key; print(generate_key().hex())"` and kept outside the repository: the run's journal is sealed with it, and a lost key is a lost journal.

**What runs:** `configs/kaggle_t4_journal.json` — 40M tokens of fine tune from `be1fa8139f59`, half the steps on the organ use curriculum (the target of every example is computed from what the journal returned: cite what was shown, else abstain), the LM arm of the frozen protocol at the end (`metrics/mqar/hm-lm-<run_id>.json`, its fields in the record), then **session B in a new process** from the checkpoint and the sealed journal alone (`metrics/mqar/hm-lm-<run_id>-session-b.json`): the named boundary of ADR-007 D8, on the model. Expected wall time: well under an hour on T4 (40M tokens is a tenth of N1's training; the curriculum roughly doubles a step). A FAIL is a result and says which of the three sub measures broke (encoder, reader, answer).

**Download the log as a file before reading any number.** Then commit: `metrics/runs.jsonl` (the record line), `metrics/LATEST.md`, `metrics/mqar/hm-lm-<run_id>.json` (the in process arm) and `metrics/mqar/hm-lm-<run_id>-session-b.json` (the new process); the checkpoint and the sealed journal stay in the Output.


In [ ]:
!git clone https://github.com/vaneeckhoutnicolas/quantum-cortex.git
%cd quantum-cortex
!pip -q install jsonschema datasets cryptography


In [ ]:
# the key: from the Kaggle secret to the environment, never to a file
import os
from kaggle_secrets import UserSecretsClient
os.environ['QUANTUM_CORTEX_JOURNAL_KEY'] = UserSecretsClient().get_secret('QUANTUM_CORTEX_JOURNAL_KEY')
assert len(bytes.fromhex(os.environ['QUANTUM_CORTEX_JOURNAL_KEY'])) == 32, 'the key must be 32 bytes hex'
print('journal key: present (not printed)')


In [ ]:
# the parent checkpoint (N1, runs/kaggle_t4/ckpt.pt) and the data slice, from the mounted Inputs
import os, shutil, glob
os.makedirs('runs/kaggle_t4', exist_ok=True); os.makedirs('data', exist_ok=True); os.makedirs('runs/journal-n1', exist_ok=True)
parent = sorted(glob.glob('/kaggle/input/**/runs/kaggle_t4/ckpt.pt', recursive=True))
assert parent, 'mount the N1 version (its Output holds runs/kaggle_t4/ckpt.pt) as an Input'
shutil.copy(parent[0], 'runs/kaggle_t4/ckpt.pt'); print('parent checkpoint:', parent[0])
own = sorted(glob.glob('/kaggle/input/**/runs/journal-n1/ckpt.pt', recursive=True))
if own: shutil.copy(own[0], 'runs/journal-n1/ckpt.pt'); print('own checkpoint recovered (resume):', own[0])
data = sorted(glob.glob('/kaggle/input/**/data/fineweb_edu_bytes.bin', recursive=True))
if data:
    shutil.copy(data[0], 'data/fineweb_edu_bytes.bin'); print('data slice reused:', data[0])
else:
    import numpy as np
    from datasets import load_dataset
    budget = 600_000_000
    out = np.memmap('data/fineweb_edu_bytes.bin', dtype=np.uint16, mode='w+', shape=(budget,))
    ds = load_dataset('HuggingFaceFW/fineweb-edu', name='sample-10BT', split='train', streaming=True)
    i = 0
    for ex in ds:
        b = ex['text'].encode('utf-8', errors='ignore'); m = min(len(b), budget - i)
        out[i:i+m] = np.frombuffer(b[:m], dtype=np.uint8); i += m
        if i >= budget: break
    out.flush(); print(f'data slice rebuilt: {i:,} byte-tokens')


In [ ]:
flag = '--resume' if os.path.exists('runs/journal-n1/ckpt.pt') else ''
!python train.py --config configs/kaggle_t4_journal.json {flag} --time-budget-min 645


In [ ]:
# the named boundary, on the model: session B in a NEW process, from the checkpoint and the sealed journal alone
!python -m cortex_c2b.hm_lm --session-b --ckpt runs/journal-n1/ckpt.pt --journal runs/journal-n1/journal.jsonl --config configs/kaggle_t4_journal.json


In [ ]:
import json
a = json.load(open('runs/journal-n1/hm-lm.json')); import glob; b = json.load(open(sorted(glob.glob('metrics/mqar/hm-lm-*-session-b.json'))[-1]))
print('in process :', a['verdict']); print('  ', {k: a[k] for k in ('hm_recall_on','hm_recall_off','hm_skill_delta','hm_invalid_citation_on','hm_false_abstention_on','hm_negctrl_rate','persistent','claimable')})
print('  attribution (encoder, reader, answer):', a['attribution'])
print('new process:', {k: b[k] for k in ('hm_recall_on','hm_recall_off','hm_invalid_citation_on','hm_false_abstention_on','hm_retrieval_hit','persistent')})
same = all(a[k] == b[k] for k in ('hm_recall_on','hm_recall_off','hm_invalid_citation_on','hm_false_abstention_on','hm_retrieval_hit'))
print('session B reproduces the in process arm:', same)
print(open('metrics/LATEST.md').read()); print(open('metrics/runs.jsonl').read().splitlines()[-1])
print('Commit: metrics/runs.jsonl, metrics/LATEST.md, metrics/mqar/hm-lm-<run_id>.json, metrics/mqar/hm-lm-<run_id>-session-b.json. Read cold first.')
